# Stage 3 — DPO Preference Alignment

Preference alignment of the Stage 2 instruction-tuned `Qwen/Qwen2.5-0.5B` on California homeowners-insurance chosen/rejected pairs using [Unsloth](https://github.com/unslothai/unsloth) + `trl`'s `DPOTrainer`, LoRA/QLoRA, on a free Colab T4.

Spec: `specs/002-unsloth-ca-homeowners-finetune.md` §6-7 (Stage 3).

**Run this notebook on Google Colab with a T4 GPU runtime** (Runtime → Change runtime type → T4 GPU). It has not been executed locally — there is no CUDA GPU in the local dev environment for this repo.

Before running: upload/clone this repo into the Colab session (`!git clone <repo-url>` then `%cd <repo-name>`), and run Stage 2 (`notebooks/2-instruction_finetuning.ipynb`) first so `models/stage2-merged/` exists — this notebook continues from that checkpoint. If it's missing, this notebook falls back to the raw base model and prints a warning.

## 0. Install dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps "trl>=0.9.0" "peft>=0.11.0" "accelerate>=0.30.0" "bitsandbytes>=0.43.0"

In [ ]:
import sys
from pathlib import Path

# Assumes this notebook is run from notebooks/ inside the cloned repo, so the
# repo root (one level up) contains the src/ package and data/ directory.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "3-preference_dataset.jsonl"
STAGE2_MODEL_DIR = REPO_ROOT / "models" / "stage2-merged"
ADAPTER_OUT_DIR = REPO_ROOT / "models" / "stage3-dpo-adapter"
MERGED_OUT_DIR = REPO_ROOT / "models" / "stage3-final-merged"
print("Repo root:", REPO_ROOT)
print("Data path exists:", DATA_PATH.exists())
print("Stage 2 merged model exists:", STAGE2_MODEL_DIR.exists())

## 1. Load the preference dataset

In [ ]:
from src.data_utils import load_jsonl

pairs = load_jsonl(DATA_PATH)
print(f"Loaded {len(pairs)} chosen/rejected pairs")
print(pairs[0])

## 2. Load the Stage 2 model (continuing from Stage 2)

Loads the Stage 2 **merged** checkpoint (base weights with Stage 1 domain adaptation + Stage 2 instruction tuning baked in) as this stage's starting "base" model, then attaches a fresh set of Stage 3 LoRA adapters below (spec §6.4).

In [ ]:
from src.model_utils import BASE_MODEL_NAME, MAX_SEQ_LENGTH, load_base_model

stage2_source = STAGE2_MODEL_DIR if STAGE2_MODEL_DIR.exists() else BASE_MODEL_NAME
if stage2_source == BASE_MODEL_NAME:
    print("Stage 2 merged model not found locally — falling back to the raw base model.")

model, tokenizer = load_base_model(
    model_name=str(stage2_source),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
print("Loaded:", stage2_source, "| eos_token:", tokenizer.eos_token)

## 3. Format the preference dataset for `DPOTrainer`

In [ ]:
from src.data_utils import build_preference_dataset

dataset = build_preference_dataset(pairs, tokenizer)
print(dataset)
print(dataset[0]["prompt"][:300])

## 4. Apply fresh LoRA / QLoRA adapters for Stage 3

The same PEFT model is used as both the policy and (with adapters disabled) the implicit reference model — passing `ref_model=None` to `DPOTrainer` below relies on this.

In [ ]:
from src.model_utils import add_lora_adapters

model = add_lora_adapters(model)  # uses DEFAULT_LORA_CONFIG: r=16, alpha=16, dropout=0.05

## 5. Configure and run `DPOTrainer`

In [ ]:
from trl import DPOConfig, DPOTrainer

dpo_args = DPOConfig(
    output_dir=str(REPO_ROOT / "outputs" / "stage3"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,  # effective batch size 4, per spec §6.3
    num_train_epochs=2,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="adamw_8bit",
    beta=0.1,
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=512,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    seed=42,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

dpo_result = dpo_trainer.train()
print(dpo_result.metrics)

## 6. Save the final Stage 3 adapter and merged model

Colab local disk is not persistent across sessions — copy these directories to Google Drive or push them to a private Hugging Face model repo after this cell (see spec §6.4). `MERGED_OUT_DIR` is the final model `src/inference.py` loads.

In [ ]:
ADAPTER_OUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_OUT_DIR))
tokenizer.save_pretrained(str(ADAPTER_OUT_DIR))
print("Saved adapter to:", ADAPTER_OUT_DIR)

In [ ]:
from src.model_utils import save_merged_model

MERGED_OUT_DIR.mkdir(parents=True, exist_ok=True)
save_merged_model(model, tokenizer, str(MERGED_OUT_DIR))
print("Saved merged model to:", MERGED_OUT_DIR)

In [ ]:
# Optional: persist to Google Drive so the adapter/merged model survive session restarts.
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# shutil.copytree(ADAPTER_OUT_DIR, "/content/drive/MyDrive/stage3-dpo-adapter", dirs_exist_ok=True)
# shutil.copytree(MERGED_OUT_DIR, "/content/drive/MyDrive/stage3-final-merged", dirs_exist_ok=True)

## 7. Run the 10 fixed evaluation questions

Same fixed question set as `reports/base_model_evaluation.md` and `reports/sft_model_comparison.md` (spec §4/§8), so answers are directly comparable. Copy these into `reports/final_evaluation.md` alongside the base and Stage 2 (SFT) answers.

In [ ]:
from unsloth import FastLanguageModel

from src.generation_utils import generate
from src.prompts import EVAL_QUESTIONS, SYSTEM_PROMPT

FastLanguageModel.for_inference(model)

for question in EVAL_QUESTIONS:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    answer = generate(model, tokenizer, prompt, max_new_tokens=200, temperature=0.7)
    print(f"Q: {question}\nA: {answer}\n{'-' * 80}")